# 03 — Anomaly Detection in Sensor Events

This notebook builds an unsupervised anomaly detection workflow for operational sensor data.

The hidden anomaly labels are used only after scoring, as an offline evaluation tool.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_sensor_anomaly_data


In [2]:
dataset = make_sensor_anomaly_data(n_points=3_000, contamination=0.04, random_state=42)
raw = dataset.features
labels = dataset.hidden_labels

raw.head()


,timestamp,temperature,motion_count,power_usage,signal_strength,missing_ratio
0,2026-01-01 00:00:00,21.182830,8.0,3.639946,-67.584958,0.049494
1,2026-01-01 00:15:00,20.376010,2.0,0.822426,-61.593665,0.100338
2,2026-01-01 00:30:00,21.450271,4.0,0.987560,-59.467281,0.063276
3,2026-01-01 00:45:00,21.564339,7.0,2.777653,-60.135436,0.216096
4,2026-01-01 01:00:00,20.476426,2.0,1.728149,-64.278383,0.002398


In [3]:
feature_columns = [
    "temperature",
    "motion_count",
    "power_usage",
    "signal_strength",
    "missing_ratio",
]

x = StandardScaler().fit_transform(raw[feature_columns])
y_true = (labels == "anomaly").astype(int).to_numpy()


## Score multiple anomaly detectors

In [4]:
scores = pd.DataFrame(index=raw.index)

isolation = IsolationForest(contamination=0.04, random_state=42)
isolation.fit(x)
scores["isolation_forest"] = -isolation.score_samples(x)

lof = LocalOutlierFactor(n_neighbors=35, contamination=0.04)
lof_labels = lof.fit_predict(x)
scores["local_outlier_factor"] = -lof.negative_outlier_factor_

robust_cov = EllipticEnvelope(contamination=0.04, random_state=42)
robust_cov.fit(x)
scores["robust_covariance"] = -robust_cov.score_samples(x)

svm = OneClassSVM(nu=0.04, gamma="scale")
svm.fit(x)
scores["one_class_svm"] = -svm.score_samples(x)

pca = PCA(n_components=2, random_state=42)
x_low = pca.fit_transform(x)
x_reconstructed = pca.inverse_transform(x_low)
scores["pca_reconstruction"] = np.mean((x - x_reconstructed) ** 2, axis=1)

scores.head()


,isolation_forest,local_outlier_factor,robust_covariance,one_class_svm,pca_reconstruction
0,0.444858,1.202720,7.186113,-11.623714,0.172471
1,0.445797,1.036837,4.090682,-12.712706,0.110287
2,0.448113,1.124723,6.726786,-12.164813,0.203478
3,0.470969,1.196191,14.195591,-11.971130,0.310426
4,0.414441,1.033800,4.982857,-11.536622,0.091086


## Precision at top-k

In [5]:
def precision_at_k(score_values, y_true, k):
    top_indices = np.argsort(score_values)[-k:]
    return y_true[top_indices].mean()

k = int(y_true.sum())
results = {
    column: precision_at_k(scores[column].to_numpy(), y_true, k)
    for column in scores.columns
}

pd.Series(results, name=f"precision_at_{k}").sort_values(ascending=False)


robust_covariance       1.000000
isolation_forest        0.991667
one_class_svm           0.508333
pca_reconstruction      0.391667
local_outlier_factor    0.050000
Name: precision_at_120, dtype: float64

## Review top anomalies

In [6]:
best_score = scores.mean(axis=1)
review = raw.copy()
review["anomaly_score"] = best_score
review["hidden_label_for_offline_eval"] = labels

review.sort_values("anomaly_score", ascending=False).head(10)


,timestamp,temperature,motion_count,power_usage,signal_strength,missing_ratio,anomaly_score,hidden_label_for_offline_eval
2421,2026-01-26 05:15:00,32.703990,3.0,7.901815,-86.278018,0.947342,144.526067,anomaly
1571,2026-01-17 08:45:00,32.985278,9.0,9.759270,-86.195896,0.903188,142.710119,anomaly
2218,2026-01-24 02:30:00,32.029555,5.0,8.268175,-79.163562,0.928147,133.726380,anomaly
118,2026-01-02 05:30:00,31.661553,9.0,8.682417,-96.410238,0.925959,133.715826,anomaly
1595,2026-01-17 14:45:00,28.156326,2.0,8.002244,-85.527260,0.875067,132.492201,anomaly
1557,2026-01-17 05:15:00,29.246266,8.0,8.883523,-90.541250,0.890974,127.527554,anomaly
2524,2026-01-27 07:00:00,33.049586,3.0,7.143708,-92.906706,0.858892,123.020152,anomaly
1465,2026-01-16 06:15:00,30.887255,8.0,8.367132,-90.595368,0.901495,122.600644,anomaly
368,2026-01-04 20:00:00,29.123424,2.0,7.006961,-82.923086,0.913995,121.500403,anomaly
16,2026-01-01 04:00:00,30.229209,7.0,9.692563,-73.555009,0.747847,119.561005,anomaly


## Limitations

Anomaly scores are not probabilities. Thresholds should be calibrated with analyst feedback, operational cost, and tolerance for false positives.
